<div dir="rtl">

# الدرس 45: نمذجة اللغة: توقّع الرمز التالي

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### المثال المحلول

</div>

In [ ]:
import math
from collections import Counter, defaultdict

sentences = ["أنا أحب القهوة", "أنا أحب الشاي", "أنت تحب القهوة"]
counts = defaultdict(Counter)
for s in sentences:
    words = ["<s>"] + s.split() + ["</s>"]
    for prev, nxt in zip(words, words[1:]):
        counts[prev][nxt] += 1

def prob(prev, nxt):
    return counts[prev][nxt] / sum(counts[prev].values())

def sentence_prob(s):
    words = ["<s>"] + s.split() + ["</s>"]
    steps = [prob(a, b) for a, b in zip(words, words[1:])]
    return math.prod(steps), steps

p, steps = sentence_prob("أنا أحب القهوة")
loss = -sum(math.log(q) for q in steps) / len(steps)
print("P =", round(p, 4), " steps:", [round(q, 3) for q in steps])
print("loss =", round(loss, 3), " perplexity =", round(math.exp(loss), 2))
print("P(أنت تحب الشاي) =", sentence_prob("أنت تحب الشاي")[0])

<div dir="rtl">

### حروف شكسبير

</div>

In [ ]:
import os
import urllib.request
import numpy as np
import torch
from torch import nn

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()

chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}           # character -> id
itos = {i: c for c, i in stoi.items()}               # id -> character
data = np.array([stoi[c] for c in text])
split = int(0.9 * len(data))
train, val = data[:split], data[split:]              # first 90% / last 10%
print("vocabulary:", V, "characters")
print(repr("".join(chars)))
print(f"train {len(train):,} | validation {len(val):,}")

<div dir="rtl">

### نموذج الثنائيات بالعدّ

</div>

In [ ]:
N = np.zeros((V, V))
np.add.at(N, (train[:-1], train[1:]), 1)              # count every (previous, next) pair
P = (N + 1) / (N + 1).sum(axis=1, keepdims=True)      # add-one smoothing, rows sum to 1

def nll(P, d):
    return -np.mean(np.log(P[d[:-1], d[1:]]))

val_loss = nll(P, val)
print(f"validation loss {val_loss:.3f} | perplexity {np.exp(val_loss):.1f} | "
      f"know-nothing baseline: loss {np.log(V):.3f}, perplexity {V}")
for c in ["q", "t", " "]:
    top = np.argsort(-P[stoi[c]])[:4]
    print(f"after {c!r}:", [(itos[j], round(float(P[stoi[c], j]), 3)) for j in top])

In [ ]:
import matplotlib.pyplot as plt
letters = [stoi[c] for c in "abcdefghijklmnopqrstuvwxyz"]
sub = P[np.ix_(letters, letters)]
plt.figure(figsize=(6.2, 5.6))
plt.imshow(sub, cmap="Blues")
plt.xticks(range(26), list("abcdefghijklmnopqrstuvwxyz"), fontsize=8)
plt.yticks(range(26), list("abcdefghijklmnopqrstuvwxyz"), fontsize=8)
plt.xlabel("next character"); plt.ylabel("previous character")
plt.colorbar(fraction=0.046)
plt.show()

<div dir="rtl">

### توليد نص

</div>

In [ ]:
rng = np.random.default_rng(0)
i = stoi["\n"]
out = []
for _ in range(250):
    i = rng.choice(V, p=P[i])
    out.append(itos[i])
print("".join(out))

<div dir="rtl">

### نفس النموذج، لكن كشبكة

</div>

In [ ]:
torch.manual_seed(0)
model = nn.Embedding(V, V)                  # row = previous char, columns = logits of next char
nn.init.zeros_(model.weight)                # start with a uniform prediction
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

x_tr, y_tr = torch.tensor(train[:-1]), torch.tensor(train[1:])
x_val, y_val = torch.tensor(val[:-1]), torch.tensor(val[1:])
with torch.no_grad():
    print(f"step    0: val loss {loss_fn(model(x_val), y_val).item():.3f}")
g = torch.Generator().manual_seed(0)
for step in range(1, 3001):
    idx = torch.randint(0, len(x_tr), (4096,), generator=g)
    loss = loss_fn(model(x_tr[idx]), y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0:
        with torch.no_grad():
            print(f"step {step:4}: val loss {loss_fn(model(x_val), y_val).item():.3f}")

<div dir="rtl">

## تمرين: ثنائيات الكلمات

استخدم الجمل الثلاث من المثال المحلول، مع التنعيم بإضافة 1، حيث الكلمات الممكنة بعد أي كلمة سبع.

1. احسب بيدك احتمال "أحب" بعد "أنا" مع التنعيم.
2. احسب احتمال جملة "أنت تحب الشاي" كاملة مع التنعيم.
3. تحقق بالكود.

</div>

In [ ]:
# اكتب حلّك هنا
